# Project Assignment 3: Advanced Data Wrangling

**Objective:**
In the previous assignment, you handled missing values and data types. Now, you will use string manipulation, merging, and reshaping to prepare your data for deep analysis.

**Guidelines:**
* **Analysis First:** Even if your data looks "clean," you must programmatically verify it. Do not assume text is consistent just by looking at the first few rows.
* **Universal Justification:** For every section, you must explain your logic. 
    * If you performed a transformation, explain *why* it helps your analysis.
    * If you *skipped* a transformation, explain *why* the current format is already perfect for your goals.
* **Reference:** Use the techniques covered in **Lecture 7** (Strings, Merging, Reshaping).

In [3]:
# --- Import Libraries ---
import pandas as pd
import numpy as np

# TODO: Load your CLEAN dataset (the CSV you saved at the end of PA2)
df_clean = pd.read_csv('../data/amazon_project_data_clean.csv')
df_sales = pd.read_csv('../data/amazon_sales_dataset.csv')
# Print column names
print("CLEAN Dataset Columns:")
print(df_clean.columns)

print("\nSALES Dataset Columns:")
print(df_sales.columns)

# Preview first 5 rows
print("\nCLEAN Dataset Preview:")
print(df_clean.head())

print("\nSALES Dataset Preview:")
print(df_sales.head())

CLEAN Dataset Columns:
Index(['order_id', 'order_date', 'product_id', 'product_category', 'price',
       'discount_percent', 'quantity_sold', 'customer_region',
       'payment_method', 'rating', 'review_count', 'discounted_price',
       'total_revenue', 'order_month'],
      dtype='str')

SALES Dataset Columns:
Index(['order_id', 'order_date', 'ship_date', 'delivery_date', 'order_status',
       'customer_id', 'customer_name', 'country', 'state', 'city',
       'product_id', 'product_name', 'category', 'sub_category', 'brand',
       'quantity', 'unit_price', 'discount', 'shipping_cost', 'total_sales',
       'payment_method'],
      dtype='str')

CLEAN Dataset Preview:
   order_id  order_date  product_id product_category   price  \
0         1  2022-04-13        2637            Books  128.75   
1         2  2023-03-12        2300          Fashion  302.60   
2         3  2022-09-28        3670           Sports  495.80   
3         4  2022-04-17        2522            Books  371.95  

## Part 1: String Analysis & Processing (1.5 Points)

Text data often contains inconsistencies (e.g., "  Phila" vs "Phila") or hidden features (e.g., extracting "2024" from "01-01-2024").

**Your Task:**
1.  **Audit:** Select a text-based column and check it for issues (whitespace, inconsistent capitalization, or mixed formats).
2.  **Process:** Use Pandas `.str` methods to clean the text OR extract a new feature (like separating First/Last name).
3.  **Alternative:** If your text data is already perfectly clean, you must **prove it** using code (e.g., checking for unique values or whitespace) and explain your verification in the markdown cell.

In [4]:
# TODO: Audit and/or Process a string column.
# Example 1 (Cleaning): df['city'] = df['city'].str.strip().str.title()
# Example 2 (Extraction): df['state'] = df['address'].str.split(',').str[-1]
# Example 3 (Verification): Check if there is trailing whitespace in a column

print("Unique Payment Methods:")
print(df_clean['payment_method'].unique())


print("\nWhitespace issues:")
print((df_clean['payment_method'] != df_clean['payment_method'].str.strip()).sum())


print("\nLowercase comparison:")
print(df_clean['payment_method'].str.lower().unique())

df_clean['payment_method_clean'] = (
    df_clean['payment_method']
        .str.strip()       # remove whitespace
        .str.lower()       # standardize case
        .str.title()       # clean formatting
)

# Verify
print(df_clean[['payment_method', 'payment_method_clean']].head())
print("\nNew unique values:")
print(df_clean['payment_method_clean'].unique())

Unique Payment Methods:
<StringArray>
['UPI', 'Credit Card', 'Wallet', 'Cash on Delivery', 'Debit Card']
Length: 5, dtype: str

Whitespace issues:
0

Lowercase comparison:
<StringArray>
['upi', 'credit card', 'wallet', 'cash on delivery', 'debit card']
Length: 5, dtype: str
  payment_method payment_method_clean
0            UPI                  Upi
1    Credit Card          Credit Card
2            UPI                  Upi
3            UPI                  Upi
4            UPI                  Upi

New unique values:
<StringArray>
['Upi', 'Credit Card', 'Wallet', 'Cash On Delivery', 'Debit Card']
Length: 5, dtype: str


**Reflection:**
*Double click here to edit.*

I audited the payment_method column to check for formatting inconsistencies such as extra whitespace and inconsistent capitalization. I first examined the unique values and verified whether any leading or trailing spaces existed by comparing the column to its stripped version. The whitespace check returned 0, indicating there were no extra spaces.

Next, I checked for capitalization differences by converting the values to lowercase and reviewing the unique outputs. The original data was already consistently formatted (e.g., “UPI”, “Credit Card”, “Wallet”, etc.), so no inconsistencies were found.

Although the data was already clean, I created a standardized payment_method_clean column using .str.strip(), .str.lower(), and .str.title() to ensure consistent formatting for future grouping or analysis. 

## Part 2: Combining and Merging Datasets (1.5 Points)

Real-world analysis often requires connecting multiple data sources. 

**Your Task:**
1.  Import a secondary dataset (e.g., weather, census data, a separate year, or a mapping key).
2.  Use `pd.merge()` or `pd.concat()` to combine it with your main dataframe.

**Note:** If your project is strictly limited to a single file, you must provide a detailed explanation below of why external data is not needed.

In [5]:
# TODO: Load a secondary dataset
df_secondary = pd.read_csv('../data/amazon_sales_dataset.csv')

# TODO: Merge or Concatenate
# df_merged = pd.merge(df, df_secondary, left_on='...', right_on='...')
# Convert both to string
df_clean['order_id'] = df_clean['order_id'].astype(str)
df_secondary['order_id'] = df_secondary['order_id'].astype(str)

# Now merge
df_merged = pd.merge(
    df_clean,
    df_secondary,
    on='order_id',
    how='left'
)

print(df_merged.head())
print("Merged shape:", df_merged.shape)

  order_id order_date_x  product_id_x product_category   price  \
0        1   2022-04-13          2637            Books  128.75   
1        2   2023-03-12          2300          Fashion  302.60   
2        3   2022-09-28          3670           Sports  495.80   
3        4   2022-04-17          2522            Books  371.95   
4        5   2022-03-13          1717           Beauty  201.68   

   discount_percent  quantity_sold customer_region payment_method_x  rating  \
0                10              4   North America              UPI     3.5   
1                20              5            Asia      Credit Card     3.7   
2                20              2          Europe              UPI     4.4   
3                15              4     Middle East              UPI     5.0   
4                 0              4     Middle East              UPI     4.6   

   ...  product_name  category  sub_category  brand quantity unit_price  \
0  ...           NaN       NaN           NaN    NaN  

**Reflection & Justification:**
*Double click here to edit.*


* **If you skipped:** Explain why your current dataset is sufficient. Why would adding external data (like weather or demographics) *not* improve your analysis?


I attempted to merge the Amazon sales dataset with my original cleaned dataset. However, neither order_id nor product_id overlapped between the two files, meaning there was no reliable shared key to perform a meaningful merge. Merging without a valid common identifier would have resulted in incorrect or unmatched records.

Additionally, my original dataset already contains all the relevant variables needed for my analysis, including pricing, discounts, revenue, customer region, payment method, ratings, and review counts. Because the dataset is already comprehensive and , adding external data would not meaningfully improve the analysis or provide additional explanatory value for this project.

## Part 3: Reshaping and Pivoting (1.0 Point)

Data is not always in the right shape for visualization. 

**Your Task:**
Perform a reshaping operation (Pivot, Melt, Stack, or Unstack) OR justify why the current shape is ideal.

* **Pivoting:** Useful for comparing categories side-by-side (e.g., making Years into columns).
* **Melting:** Useful for visualization libraries (e.g., collapsing multiple "Month" columns into a single "Date" column).

In [6]:
# TODO: Reshape your data
# df_pivoted = df.pivot(...)
# OR
# df_melted = pd.melt(...)

pivot_table = df_clean.pivot_table(
    values='total_revenue',
    index='product_category',
    columns='order_month',
    aggfunc='sum'
)

print(pivot_table.head())

order_month              1          2          3          4          5   \
product_category                                                          
Beauty            475107.17  382718.88  471533.93  496134.02  511285.10   
Books             451515.08  440442.82  456431.77  436442.33  465133.15   
Electronics       478311.78  448082.99  441314.67  450272.92  462965.61   
Fashion           503112.82  417628.62  463734.34  451558.58  465934.08   
Home & Kitchen    479790.88  415405.08  420652.95  438411.94  450406.17   

order_month              6          7          8          9          10  \
product_category                                                          
Beauty            448008.63  449906.31  483284.92  480709.01  444146.19   
Books             466860.42  445647.31  454348.96  505484.30  471752.91   
Electronics       437399.22  474250.16  444605.32  459050.35  464082.23   
Fashion           430175.20  480504.66  458640.04  425470.82  467513.14   
Home & Kitchen    519730

**Reflection & Justification:**
*Double click here to edit.*

* **If you reshaped:** How does this new structure make it easier to plot or analyze the data?
I reshaped the dataset using a pivot table so that product_category appears as rows and order_month appears as columns, with total revenue aggregated as the values. This new structure makes it easier to compare revenue performance across months for each product category side-by-side. Instead of filtering and grouping repeatedly, the pivoted format provides a clear summary table that can quickly be used for bar charts, heatmaps, or trend comparisons across categories.


## Part 4: Final View (1.0 Point)

Display the head and info of your final, processed dataframe to demonstrate the changes.

In [7]:
# TODO: Display .head() and .info()
print("Final Dataset Preview:")
print(df_clean.head())

print("\nFinal Dataset Info:")
print(df_clean.info())

Final Dataset Preview:
  order_id  order_date  product_id product_category   price  discount_percent  \
0        1  2022-04-13        2637            Books  128.75                10   
1        2  2023-03-12        2300          Fashion  302.60                20   
2        3  2022-09-28        3670           Sports  495.80                20   
3        4  2022-04-17        2522            Books  371.95                15   
4        5  2022-03-13        1717           Beauty  201.68                 0   

   quantity_sold customer_region payment_method  rating  review_count  \
0              4   North America            UPI     3.5           443   
1              5            Asia    Credit Card     3.7           475   
2              2          Europe            UPI     4.4           183   
3              4     Middle East            UPI     5.0           212   
4              4     Middle East            UPI     4.6           308   

   discounted_price  total_revenue  order_month pay

In [8]:
# TODO: Save your processed dataset to a new CSV file for the next assignment
df_clean.to_csv('../data/project_data_processed.csv', index=False)